# Settings and Config with Pydantic

This notebook covers:

1. Declare app config as a `pydantic_settings.BaseSettings` subclass with typed fields
2. Load values from environment variables and a `.env` file, with documented precedence
3. Inject the settings object via `Depends(get_settings)` and cache it with `@lru_cache`
4. Fail fast at startup when required config is missing or invalid
5. Compose nested settings (e.g., DB settings as a sub-model)

**Scope**: FastAPI + `pydantic-settings` + `TestClient`. We'll write and read a temporary `.env` file from inside the notebook to demonstrate file-based loading.

`pydantic_settings.BaseSettings` is Pydantic's BaseModel with a configurable source chain. Instead of `__init__` arguments, it reads from environment variables, `.env`, secrets files, and CLI args. You get the same v2 type coercion and validation as any other Pydantic model — including the "fail loudly on a wrong type" guarantee — applied to the things that go wrong in production: a missing `DATABASE_URL`, a typo in a feature flag, a string where you expected an int.

## 1. Why Typed Config

Untyped config rots. Three failure modes that compound:

- **`os.environ["LOG_LEVEL"]` is always a string.** Comparing it to `"DEBUG"` works; comparing it to `logging.DEBUG` doesn't. Every read site has to remember to cast.
- **Missing keys explode at the point of first use.** If `STRIPE_API_KEY` isn't set, you find out at 3 AM when the first payment endpoint runs, not at startup.
- **Defaults drift across modules.** Three different `os.environ.get("TIMEOUT", "30")` calls in three files — change the policy, find them all.

`BaseSettings` fixes all three:

- **Types are declared once and enforced.** `int`, `bool`, `Path`, `AnyUrl`, custom validators — all the v2 toolbox.
- **Missing-required fails at construction.** That's startup (notebook 8.1), not first request.
- **One object, one source of truth.** Code reads `settings.timeout`, not `os.environ.get(...)`.

## 2. `BaseSettings` Basics

A `BaseSettings` class looks like a Pydantic `BaseModel` — same field syntax, same validators. The difference is **where the values come from**. By default they come from environment variables (case-insensitive, matched to field name).

In [ ]:
import os
from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    app_name: str = "PortfolioAPI"
    debug: bool = False
    default_page_size: int = Field(default=20, ge=1, le=100)
    currency: str = "USD"

# Inject env vars in this process to demonstrate loading (real apps inherit them from the shell / orchestrator).
os.environ["APP_NAME"] = "PortfolioAPI-staging"
os.environ["DEBUG"] = "true"        # string "true" is coerced to bool by Pydantic
os.environ["DEFAULT_PAGE_SIZE"] = "50"

settings = Settings()
print(settings)
print("debug is a real bool:", type(settings.debug).__name__, settings.debug)
print("page size is a real int:", type(settings.default_page_size).__name__, settings.default_page_size)

Three properties to lock in:

- **Field names map to env vars uppercased.** `app_name` ← `APP_NAME`. Case-insensitive matching can be configured but this default is what almost everyone uses.
- **Strings get coerced.** `"true"` → `True`, `"50"` → `50`. No manual casting at the call site.
- **Validation runs as usual.** Try `os.environ["DEFAULT_PAGE_SIZE"] = "9999"` — it'll raise at `Settings()` time, because the field has `le=100`. Loudly, with a Pydantic error pointing at the bad field. That's the "fail fast" property we want.

## 3. Loading from `.env`

Twelve-factor config says env vars; in dev, that translates to a `.env` file the app reads at startup. `BaseSettings` reads it via the `env_file` config option. Precedence (highest wins):

1. Explicit `__init__` arguments — `Settings(debug=True)`.
2. Environment variables — `DEBUG=true` in the shell.
3. The `.env` file — `DEBUG=true` in `.env`.
4. Field defaults — declared in the class.

So a developer can set `DEBUG=true` once in `.env` and forget about it; CI can override with `DEBUG=false` via a shell variable; and an integration test can override both with `Settings(debug=True)`.

In [ ]:
from pathlib import Path

# Write a temporary .env for this demo. Real apps just commit the layout (not values).
env_path = Path(".env.demo")
env_path.write_text(
    "APP_NAME=PortfolioAPI-from-env-file\n"
    "CURRENCY=EUR\n"
)

class SettingsWithEnv(BaseSettings):
    model_config = SettingsConfigDict(env_file=str(env_path), env_file_encoding="utf-8")

    app_name: str = "PortfolioAPI"
    debug: bool = False
    default_page_size: int = 20
    currency: str = "USD"

# Clear the env vars so the .env values aren't shadowed.
for k in ["APP_NAME", "CURRENCY"]:
    os.environ.pop(k, None)

s = SettingsWithEnv()
print("loaded from .env:", s)

# Demonstrate precedence: env var wins over .env.
os.environ["CURRENCY"] = "JPY"
s2 = SettingsWithEnv()
print("env var overrides .env:", s2.currency)

# Explicit constructor arg wins over everything.
s3 = SettingsWithEnv(currency="GBP")
print("ctor arg overrides env var:", s3.currency)

## 4. Nested Settings

For non-trivial apps the config is grouped: database settings, auth settings, cache settings. You can express that with sub-models. The env-var convention uses `__` (double underscore) as a separator: `DATABASE__URL` maps to `Settings.database.url`.

In [ ]:
from pydantic import BaseModel

class DatabaseSettings(BaseModel):
    url: str = "sqlite:///./app.db"
    pool_size: int = 5
    echo: bool = False

class GroupedSettings(BaseSettings):
    model_config = SettingsConfigDict(env_nested_delimiter="__")
    app_name: str = "PortfolioAPI"
    database: DatabaseSettings = DatabaseSettings()

# Override the nested fields via env vars with __ separator.
os.environ["DATABASE__URL"] = "postgresql://localhost/portfolio"
os.environ["DATABASE__POOL_SIZE"] = "20"

gs = GroupedSettings()
print("app_name:", gs.app_name)
print("database.url:", gs.database.url)
print("database.pool_size:", gs.database.pool_size)
print("database.echo (still default):", gs.database.echo)

The grouped form scales much better than a flat `Settings` with thirty fields. Each module that needs DB config takes `database: DatabaseSettings` — not the whole settings object. Auth code takes `auth: AuthSettings`. The blast radius of a config change is the sub-model, not the whole app.

## 5. Injecting Settings via `Depends` (+ `@lru_cache`)

Building a `Settings()` object reads env vars, parses `.env`, runs validators. Doing that **on every request** is wasteful and not what anyone wants. Cache it once per process with `functools.lru_cache(maxsize=1)`.

The pattern:

```python
@lru_cache(maxsize=1)
def get_settings() -> Settings:
    return Settings()

@app.get("/health")
def health(settings: Settings = Depends(get_settings)):
    return {"app": settings.app_name}
```

`@lru_cache` makes `get_settings()` effectively a singleton: the first call constructs `Settings()`, subsequent calls return the same instance. Combined with `Depends(get_settings)`, the route receives that instance — and tests can override `get_settings` to inject a different `Settings` (see [[feedback-curriculum-conventions]] for the testing pattern). This is the standard idiom for FastAPI settings.

In [ ]:
from functools import lru_cache
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

# Clean slate of env vars for this section.
for k in ["APP_NAME", "DEBUG", "DEFAULT_PAGE_SIZE", "CURRENCY", "DATABASE__URL", "DATABASE__POOL_SIZE"]:
    os.environ.pop(k, None)
os.environ["APP_NAME"] = "PortfolioAPI-prod"

class AppSettings(BaseSettings):
    app_name: str = "PortfolioAPI"
    debug: bool = False
    default_page_size: int = 20

@lru_cache(maxsize=1)
def get_settings() -> AppSettings:
    print("(building settings — should print once per process)")
    return AppSettings()

app = FastAPI()

@app.get("/config")
def read_config(settings: AppSettings = Depends(get_settings)):
    return {"app_name": settings.app_name, "debug": settings.debug}

client = TestClient(app)
print("request 1:", client.get("/config").json())
print("request 2:", client.get("/config").json())
print("request 3:", client.get("/config").json())
# Note: the "(building settings...)" line above should appear ONLY ONCE.

You should see `(building settings...)` print exactly once, even across multiple requests. `@lru_cache` keyed on no arguments collapses to "always the same call" → "always the same result".

The same override pattern from notebook 4.2 applies to settings. In a test, install a custom `AppSettings` via `app.dependency_overrides[get_settings] = lambda: AppSettings(debug=True)` — every route that depends on settings sees the test config without touching the rest of the app.

## 6. Fail-Fast Validation at Startup

The point of typed config is to detect bad config **before any traffic arrives** — at process startup, with a clear error message. The pattern is:

- Mark required fields with no default (so Pydantic raises if they're missing).
- Use field validators / constraints to express domain rules (`min_length`, `pattern`, `ge`, custom `@field_validator`).
- Call `get_settings()` from the lifespan startup hook (notebook 8.1) so failures crash the worker on boot.

If `Settings()` raises, uvicorn never starts, your health check stays red, and your orchestrator (Kubernetes / Compose / systemd) marks the container failed. Better than serving 500s from a half-configured app.

In [ ]:
from pydantic import ValidationError, SecretStr

class StrictSettings(BaseSettings):
    database_url: str = Field(min_length=1)  # required: no default
    api_key: SecretStr                       # required: no default; SecretStr won't leak via repr
    log_level: str = Field(default="INFO", pattern=r"^(DEBUG|INFO|WARNING|ERROR)$")

# Case 1: missing required field -> clear error.
try:
    StrictSettings(database_url="", api_key="")
except ValidationError as e:
    print("missing-required error (truncated):")
    print(str(e)[:300], "...")

# Case 2: bad enum-like value -> clear error pointing at the field.
try:
    StrictSettings(database_url="sqlite://", api_key="x", log_level="VERBOSE")
except ValidationError as e:
    print("\nbad value error (truncated):")
    print(str(e)[:300], "...")

# Case 3: happy path. SecretStr keeps the secret out of repr / logs.
s = StrictSettings(database_url="postgres://...", api_key="super-secret", log_level="DEBUG")
print("\nhappy:", s)
print("api_key.get_secret_value():", s.api_key.get_secret_value())

Two notes on the safety helpers used above:

- **`Field(min_length=1)` (or simply omitting a default)** forces the env var to be present and non-empty. The error message tells you exactly which field is missing.
- **`SecretStr`** prevents accidental logging. `print(s)` shows `api_key=SecretStr('**********')`. The actual value is only accessible via `.get_secret_value()` — a deliberate ceremony that makes the leak visible at every call site.

## Key Takeaways

- **Typed config = `BaseSettings`.** Same v2 toolbox as `BaseModel`, applied to env vars, `.env`, and secrets.
- **Precedence (high → low)**: init args → env vars → `.env` → defaults. Memorize this — it explains every "why is the wrong value loading?" debugging session.
- **Inject with `Depends(get_settings)` + `@lru_cache`** for a process-wide singleton that tests can still override via `app.dependency_overrides`.
- **Nested settings** with `env_nested_delimiter="__"` keep the config object navigable as it grows.
- **Fail fast.** Required fields without defaults; call `get_settings()` in lifespan startup. A broken config crashes the boot, not the first request.
- **`SecretStr`** for keys and tokens — leaks via `repr` and naive logging become impossible.
- **Capstone tie-in**: the portfolio API will have `AppSettings(database: ..., auth: ..., observability: ...)`, instantiated once in lifespan startup, injected everywhere via `Depends`.

## Exercises

**1. Per-env feature flag.** Add a `features: dict[str, bool] = {"sse_prices": False, "background_export": False}` field to `AppSettings`. Override `FEATURES__SSE_PRICES=true` via env var and confirm only that flag flips. Now write a route that returns `{"sse_enabled": settings.features["sse_prices"]}` and verify it follows the env var.

**2. Required at startup.** Define `class ProdSettings(BaseSettings)` with `database_url: str` (no default) and `api_key: SecretStr` (no default). In a `lifespan` startup function (preview of 8.1), call `ProdSettings()` and catch the `ValidationError`. Make the app refuse to start if it fires. Now test both paths: with the env vars set (clean startup) and without (refuses).

**3. Override in a test, then restore.** Using the override pattern from 4.2, swap `get_settings` for a callable that returns `AppSettings(debug=True, default_page_size=5)`. Hit `/config` and confirm the override took effect. Clear the override; hit `/config` again; confirm you're back to production values. This is the same machinery every settings-using test in chapter 7 will use.